# Module 4: SQL-Based Analysis

## Overview

This notebook generates business insights from the cleaned retail transaction dataset using SQL queries.

It produces analytical reports such as Monthly Revenue Analysis, Top Customers, Payment Method Distribution, and Product Category Performance. These reports help organizations understand sales trends and customer behavior.

In [0]:
# Load Clean Layer

from pyspark.sql.functions import *

clean_df = spark.table("trustguard.clean_transactions")

display(clean_df.limit(10))

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
TXN_8425168,CUST_25,Milk Products,Item_16_MILK,27.5,7,192.5,Cash,In-store,2022-03-12,false
TXN_5885894,CUST_01,Patisserie,Item_14_PAT,24.5,10,245.0,Credit Card,In-store,2022-02-09,false
TXN_3671271,CUST_18,Patisserie,Item_12_PAT,21.5,8,172.0,Cash,In-store,2023-11-27,false
TXN_9908330,CUST_19,Food,Item_18_FOOD,30.5,4,122.0,Credit Card,Online,2022-09-29,true
TXN_2780662,CUST_02,Milk Products,Item_19_MILK,32.0,10,320.0,Digital Wallet,In-store,2023-01-19,false
TXN_2646301,CUST_18,Furniture,Item_24_FUR,39.5,9,355.5,Credit Card,In-store,2022-12-07,false
TXN_5328604,CUST_07,Food,Item_20_FOOD,33.5,10,335.0,Cash,Online,2024-07-08,false
TXN_8219228,CUST_17,Computers And Electric Accessories,Item_10_CEA,18.5,5,92.5,Cash,Online,2022-05-27,false
TXN_1112365,CUST_19,Food,Item_24_FOOD,39.5,10,395.0,Cash,Online,2023-09-24,false
TXN_2953434,CUST_25,Furniture,Item_25_FUR,41.0,10,410.0,Credit Card,In-store,2023-08-10,false


In [0]:
# Calculate statistics for quantity

stats = clean_df.selectExpr(
    "avg(quantity) as mean_qty",
    "stddev(quantity) as std_qty"
).collect()[0]

mean_qty = stats["mean_qty"]
std_qty = stats["std_qty"]

print("Mean Quantity :", mean_qty)
print("Standard Deviation :", std_qty)

Mean Quantity : 5.510616302186879
Standard Deviation : 2.790756127805604


In [0]:
# Find invalid price values

invalid_price = clean_df.filter(col("price_per_unit") <= 0)

print("Invalid Price Records:", invalid_price.count())

display(invalid_price)

Invalid Price Records: 0


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied


In [0]:
# Flag anomalies

anomaly_log = clean_df.withColumn(
    "reason",
    when(
        col("quantity") > (mean_qty + 3 * std_qty),
        "Quantity greater than 3 Standard Deviations"
    ).when(
        col("price_per_unit") <= 0,
        "Invalid Unit Price"
    )
).filter(col("reason").isNotNull())

display(anomaly_log)

transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,reason


In [0]:
# Save anomaly records

anomaly_log.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("trustguard.anomaly_log")

print("Anomaly Log Saved Successfully")

Anomaly Log Saved Successfully
